# 第8回：クラスを予測する—分類

**今日の問い：正解率だけで十分なのはどんなときか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。`DEEP DIVE`は経験者や自習向けの発展です。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 混同行列とprecision・recall・F1・PR-AUCを利用場面へ結びつける
- 確率の較正（calibration）を信頼度図と指標で評価する
- 不均衡データへclass_weightや閾値調整で対処し、効果を検証する

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- precision：陽性予測のうち正しかった割合
- recall：実際の陽性を見つけた割合
- PR-AUC：適合率-再現率曲線の下側面積
- 較正：予測確率と実際の頻度が一致している度合い
- class_weight：少数クラスの誤りを重く扱う設定

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
model = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
probability = model.predict_proba(X_valid)[:, 1]


## TRY：閾値0.5で混同行列を読む


In [ ]:
prediction = (probability >= 0.5).astype(int)
print("accuracy:", round(accuracy_score(y_valid, prediction), 3))
print("precision:", round(precision_score(y_valid, prediction), 3))
print("recall:", round(recall_score(y_valid, prediction), 3))
print("F1:", round(f1_score(y_valid, prediction), 3))
ConfusionMatrixDisplay.from_predictions(y_valid, prediction, display_labels=["非活性", "活性"], cmap="Blues")
plt.title("混同行列")


## TRY：判定閾値を変える


In [ ]:
rows = []
for threshold in [0.3, 0.5, 0.7]:
    pred = (probability >= threshold).astype(int)
    rows.append({"閾値": threshold, "precision": precision_score(y_valid, pred), "recall": recall_score(y_valid, pred), "F1": f1_score(y_valid, pred)})
pd.DataFrame(rows).round(3)


## CORE深掘り：不均衡ではPR-AUCを見る

陰性が多いデータではaccuracyが高く見えます。適合率-再現率曲線の面積（PR-AUC）が実態を映します。


In [ ]:
from sklearn.metrics import average_precision_score
print("活性の割合:", round(df["active"].mean(), 3))
print("PR-AUC(平均適合率):", round(average_precision_score(y_valid, probability), 3))
print("常に多数派と予測したときのaccuracy:", round((y_valid == y_valid.mode()[0]).mean(), 3))


## 話し合い

見逃したくない探索段階ならrecall、追試コストが高い絞り込み段階ならprecision。正解は利用場面で変わります。


## DEEP DIVE：確率の較正・コスト最適閾値・不均衡対策

確率をコスト計算へ使うなら、まず較正（予測確率と実頻度の一致）が前提です。


In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

base_clf = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
cal_clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5).fit(X_train, y_train)
plt.figure(figsize=(6, 5))
for name, clf in {"未較正": base_clf, "較正後": cal_clf}.items():
    p = clf.predict_proba(X_valid)[:, 1]
    print(f"{name}: Brier={brier_score_loss(y_valid, p):.3f}（小さいほど良い）")
    frac, mean_pred = calibration_curve(y_valid, p, n_bins=5)
    plt.plot(mean_pred, frac, "o-", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("予測確率"); plt.ylabel("実際の頻度"); plt.legend(); plt.title("信頼度図")
plt.tight_layout()


### コスト行列で閾値を決める


In [ ]:
import numpy as np
proba_cal = cal_clf.predict_proba(X_valid)[:, 1]
cost_fn, cost_fp = 8, 1
rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred = (proba_cal >= t).astype(int)
    fp = int(((pred == 1) & (y_valid == 0)).sum())
    fn = int(((pred == 0) & (y_valid == 1)).sum())
    rows.append({"閾値": round(t, 2), "偽陽性": fp, "偽陰性": fn, "期待コスト": fp * cost_fp + fn * cost_fn})
table = pd.DataFrame(rows)
print("コスト最小の閾値:", table.loc[table["期待コスト"].idxmin(), "閾値"])
table


### class_weightで少数クラスを重くする


In [ ]:
for label, weight in {"weightなし": None, "balanced": "balanced"}.items():
    clf = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000, class_weight=weight)).fit(X_train, y_train)
    pred = clf.predict(X_valid)
    print(f"{label:10s} F1={f1_score(y_valid, pred):.3f}  recall={recall_score(y_valid, pred):.3f}")


## よくある誤り

- 常に閾値0.5を使う
- 偽陽性と偽陰性のコストを同じとみなす
- 未較正の確率をそのまま意思決定へ使う

## SELF-STUDY（任意・30〜60分）

- CalibratedClassifierCVで較正前後の信頼度図とBrierスコアを比べる
- コスト行列から期待コスト最小の閾値を求め、0.5と比較する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. accuracyが危険な例は何か
2. 較正が悪い確率を使うと何が起きるか
3. コストから閾値をどう決めるか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
